# Can a Computer Tell if a Bill is About China?

In this notebook you'll work with a real machine-learning model that reads
the text of congressional bills and predicts whether each one is about China.
You'll probe it, try to fool it, and race it against a much dumber competitor.

**No coding experience needed.** Almost everything is pre-written. In a few marked
spots you'll edit a line of real Python — a sentence, a number, or a list — and watch
what changes.

---

### ⚠️ Step zero — this is YOUR copy

This notebook is your personal copy — it has your name on the file. Your edits save
automatically; nobody else is working in it. If it ever gets hopelessly tangled,
**Ctrl+Z / Cmd+Z inside a cell fixes almost everything** — and if it doesn't, email
Heagen for a fresh copy.

### How this notebook works

- Cells run **top to bottom**. Click a cell, then press **Shift+Enter** (or the ▶ button) to run it.
- Cells marked **▶️ JUST RUN**: run them, don't edit them.
- Cells marked **✏️ YOUR TURN**: these have a line for you to change, marked with `# <-- EDIT`.
- Under every ✏️ cell there's a **"What you should see"** box. If your output looks
  roughly like that, you're on track. If you get a red error instead, press
  **Ctrl+Z / Cmd+Z** inside the cell to undo, and check the hint in the box.

**Time:** this notebook is built for about 4 hours total, spread however you like.
Rough guide: Setup + meet the data ~30 min · Beat 1 ~45 min · Beat 2 ~45 min ·
Beat 3 ~60 min · Bonus ~45 min.

## Setup

**▶️ JUST RUN** the next cell. It installs one library and loads our tools.
It takes a minute or two. You'll see a lot of text scroll by. That's normal.
Wait for the green checkmark on the left of the cell before moving on.

In [ ]:
# ▶️ JUST RUN — installs and imports (takes 1-2 minutes)
%pip install -q sentence-transformers

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer

pd.set_option("display.max_colwidth", 120)
print("Setup complete ✅")

## Meet the data

Our dataset is a collection of real-style congressional bills. Each row has the
bill's title, a snippet of its text, and a label a human gave it.

**▶️ JUST RUN:**

In [ ]:
# ▶️ JUST RUN — load the bills
DATA_URL = "https://raw.githubusercontent.com/heagenb03/twl-hs-interns-notebooks/main/hs-interns/data/bills_china.csv"

bills = pd.read_csv(DATA_URL)
print(f"Loaded {len(bills)} bills.")
bills.head()

> **What you should see:** a line saying `Loaded 90 bills.` and a table with 5 rows
> and 4 columns: `bill_id`, `title`, `snippet`, `about_china`.

`.head()` is a DataFrame method: it shows the first 5 rows of a table. Let's ask a
different question: how many bills of each kind do we have?

**▶️ JUST RUN:**

In [ ]:
# ▶️ JUST RUN — how many of each label?
bills["about_china"].value_counts()

> **What you should see:** two numbers — roughly half the bills are labeled `1`
> (about China) and half `0` (not).

Take two minutes and actually **read a few snippets** in the table above. Notice that
some "about China" bills never say the word *China* at all. Keep that in mind.
It's going to matter a lot in Beat 3.

## Build the model

Here's the plan: a *language model* turns each bill's text into a long
list of numbers that captures its **meaning** (so "Beijing" and "Chinese government"
end up with similar numbers). Then a simple *classifier* learns which number-patterns
go with the "about China" label.

We train it on most of the bills and hold some back as a **test set** — bills the
model has never seen — to check whether it actually learned anything.

**▶️ JUST RUN** (takes ~1 minute the first time, since it's downloading the language model):

In [ ]:
# ▶️ JUST RUN — build and train the classifier
embedding_model_name = "all-MiniLM-L6-v2"

embedder = SentenceTransformer(embedding_model_name)
X = embedder.encode(list(bills["snippet"]), show_progress_bar=True)
y = bills["about_china"].values

X_train, X_test, y_train, y_test, bills_train, bills_test = train_test_split(
    X, y, bills, test_size=0.3, random_state=42, stratify=y
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

accuracy = model.score(X_test, y_test)
print(f"Model trained on {len(X_train)} bills.")
print(f"On {len(X_test)} bills it has NEVER seen, it gets {accuracy:.0%} right.")

> **What you should see:** a progress bar, then two lines ending with an accuracy
> percentage — likely somewhere in the 80–95% range.

That percentage is nice, but it's just a number. The interesting question is *how*
the model gets things right and wrong, and that's what the rest of this notebook
is about.

**▶️ JUST RUN** this last setup cell. It defines the helper tools the rest of the
notebook uses. (You don't need to read it — but you can!)

In [ ]:
# ▶️ JUST RUN — helper tools for the rest of the notebook
def china_probability(text):
    """Return the model's confidence (0 to 1) that `text` is about China."""
    vec = embedder.encode([text])
    return model.predict_proba(vec)[0][1]

def show_prediction(text):
    """Print the model's verdict on one sentence."""
    prob = china_probability(text)
    verdict = "ABOUT CHINA 🇨🇳" if prob >= 0.5 else "not about China"
    print(f'Sentence:   "{text}"')
    print(f"Confidence: {prob:.1%}")
    print(f"Verdict:    {verdict}")

def show_borderline_bills(threshold):
    """Show the bills the model is least sure about, judged at `threshold`."""
    threshold = float(threshold)  # forgives "0.7" typed with quotes
    probs = model.predict_proba(X)[:, 1]
    table = bills.copy()
    table["confidence"] = probs
    table["distance_from_middle"] = abs(probs - 0.5)
    table = table.sort_values("distance_from_middle").head(12)
    table["verdict at your threshold"] = np.where(
        table["confidence"] >= threshold, "ABOUT CHINA", "not about China"
    )
    total = (probs >= threshold).sum()
    print(f"With threshold = {threshold}, the model calls {total} of {len(bills)} bills 'about China'.")
    print("Here are the 12 bills the model is LEAST sure about:")
    return table[["title", "confidence", "verdict at your threshold"]].round(3)

def keyword_says_china(text, terms):
    """The 'dumb' competitor: says yes if any term appears in the text."""
    lowered = text.lower()
    return any(term.lower() in lowered for term in terms)

TRICKY_SENTENCES = [
    ("To counter influence operations by the Chinese Communist Party.", 1),
    ("To strengthen deterrence in the Indo-Pacific region.", 1),
    ("To reduce reliance on foreign adversaries for advanced semiconductors.", 1),
    ("To assess the military balance across the Taiwan Strait.", 1),
    ("To restrict apps owned by ByteDance on government devices.", 1),
    ("To require disclosure of supply chains sourcing polysilicon from Xinjiang.", 1),
    ("To respond to the national security law imposed on a special administrative region.", 1),
    ("To monitor manipulation of the renminbi exchange rate.", 1),
    ("To expand broadband access in rural communities.", 0),
    ("To establish safety standards for imported porcelain china tableware.", 0),
    ("To implement the trade agreement between the United States and Japan.", 0),
    ("To fund lead pipe replacement in public water systems.", 0),
]

def race_the_models(terms):
    """Compare your keyword list against the ML model on the tricky sentences."""
    if isinstance(terms, str):
        terms = [terms]  # forgives a bare string instead of a list
    rows, kw_score, ml_score = [], 0, 0
    for text, truth in TRICKY_SENTENCES:
        kw = int(keyword_says_china(text, terms))
        ml = int(china_probability(text) >= 0.5)
        kw_score += (kw == truth)
        ml_score += (ml == truth)
        mark = lambda guess: "✅" if guess == truth else "❌"
        rows.append([text[:70], "China" if truth else "not",
                     f"{'China' if kw else 'not'} {mark(kw)}",
                     f"{'China' if ml else 'not'} {mark(ml)}"])
    result = pd.DataFrame(rows, columns=["sentence", "truth", "keyword model", "ML model"])
    print(f"FINAL SCORE — Keyword model: {kw_score}/{len(TRICKY_SENTENCES)}   ML model: {ml_score}/{len(TRICKY_SENTENCES)}")
    return result

def show_test_bills():
    """Numbered list of the held-out test bills and their human labels."""
    table = bills_test.reset_index(drop=True).copy()
    table["human label"] = np.where(table["about_china"] == 1, "about China", "not")
    return table[["title", "human label"]]

def show_confusion_matrix(flip_these):
    """Confusion matrix on the test set, after flipping the human labels at the given row numbers."""
    labels = y_test.copy()
    for i in flip_these:
        i = int(i)  # forgives "7" typed with quotes
        if 0 <= i < len(labels):
            labels[i] = 1 - labels[i]
        else:
            print(f"(Skipping {i} — row numbers go from 0 to {len(labels) - 1}.)")
    preds = model.predict(X_test)
    tp = ((preds == 1) & (labels == 1)).sum()
    tn = ((preds == 0) & (labels == 0)).sum()
    fp = ((preds == 1) & (labels == 0)).sum()
    fn = ((preds == 0) & (labels == 1)).sum()
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    matrix = np.array([[tn, fp], [fn, tp]])
    ax.imshow(matrix, cmap="Blues", vmin=0)
    texts = [["Both agree:\nNOT China", "Model says China,\nhuman said not"],
             ["Model missed it,\nhuman said China", "Both agree:\nabout China"]]
    for r in range(2):
        for c in range(2):
            ax.text(c, r, f"{matrix[r, c]}\n{texts[r][c]}", ha="center", va="center", fontsize=10)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["model: not China", "model: China"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["human: not", "human: China"])
    flips = f" (with {len(flip_these)} human label(s) flipped)" if flip_these else ""
    ax.set_title(f"Model vs. human labels on {len(labels)} unseen bills{flips}")
    plt.tight_layout(); plt.show()
    agree = tp + tn
    print(f"Model and human labels agree on {agree} of {len(labels)} bills.")

def rebuild_with(new_model_name):
    """Stretch goal: retrain everything with a different embedding model."""
    global embedder, model, X, X_train, X_test, y_train, y_test, bills_train, bills_test
    embedder = SentenceTransformer(new_model_name)
    X = embedder.encode(list(bills["snippet"]), show_progress_bar=True)
    X_train, X_test, y_train, y_test, bills_train, bills_test = train_test_split(
        X, bills["about_china"].values, bills, test_size=0.3, random_state=42,
        stratify=bills["about_china"].values)
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    print(f"Rebuilt with '{new_model_name}'. New accuracy on unseen bills: {model.score(X_test, y_test):.0%}")

print("Helpers loaded ✅ — on to Beat 1!")

---
## Beat 1 — Type your own sentence ✏️

Time to talk to the model directly. The cell below feeds it one sentence and prints
its confidence that the sentence is about China.

**✏️ YOUR TURN:** change the text between the quotation marks, then run the cell.
Keep the quotation marks!

In [ ]:
my_sentence = "A bill to expand rural broadband access."   # <-- EDIT the text inside the quotes

show_prediction(my_sentence)

> **What you should see** (with the starting sentence):
> ```
> Sentence:   "A bill to expand rural broadband access."
> Confidence: 3.1%
> Verdict:    not about China
> ```
> Your exact percentage may differ by a few points — that's normal.
> **If you get a red error:** you probably deleted a quotation mark. Press Ctrl+Z / Cmd+Z and try again.

### Things to try

Run the cell again after each edit: each run is one experiment.

1. Something obviously about China: `"Sanctions on the Chinese Communist Party."`
2. Something obviously not: `"Free ice cream in school cafeterias."`
3. **Be sneaky.** Can you write a sentence that IS about China
   without using the words *China* or *Chinese*? Try `"Deterrence in the Indo-Pacific."`
   or `"Restrictions on apps owned by ByteDance."`, then invent your own.
4. Can you go the other way: write a sentence that's NOT about China but fools the
   model into high confidence anyway?

**Jot down** (in a doc or on paper) your best "fooled it" sentence in each direction.
You'll want them for Beat 3.

**One more question:** the model never gives you 0% or 100%, just how sure it is.
So who decides where "sure enough" is? That's Beat 2.

---
## Beat 2 — You draw the line ✏️

The model gives every bill a confidence score between 0 and 1. Turning that score
into a yes/no verdict requires a cutoff: a **threshold**. Score above it: "about
China." Below: "not."

So far we've silently used 0.5. But 0.5 isn't sacred — it's a choice. And for the
bills the model is unsure about, that choice *is* the verdict.

**✏️ YOUR TURN:** change the threshold number and run the cell. Watch the borderline
bills flip.

In [ ]:
threshold = 0.5   # <-- EDIT this number (try 0.3, then 0.7, then 0.9)

show_borderline_bills(threshold)

> **What you should see** (with 0.5): a line like
> `With threshold = 0.5, the model calls 46 of 90 bills 'about China'.`
> followed by a 12-row table of bill titles with confidence scores hovering near
> 0.4–0.6, each with a verdict.
> **If you get a red error:** the threshold must be a plain number like `0.7` —
> no quotes around it.

### Things to try

1. Drop it to `0.3`. Which bills just became "about China"? Read their titles. Do
   you agree with the new verdicts?
2. Raise it to `0.9`. Now the model only flags bills it's *very* sure about. What
   gets lost?
3. Find the threshold where the count of "about China" bills changes fastest: that's
   where the model's uncertainty lives.

### Why anyone cares

Imagine two real users of this model. A **journalist** compiling a complete list of
every China-related bill would rather include a few false alarms than miss one:
they want a *low* threshold. A **hedge fund** auto-trading on the output can't afford
false alarms: they want a *high* one. Same model, same scores, different threshold,
because the **cost of being wrong** is different in each direction.

Which user would you rather be? Set the threshold for them and look at what their
list gets right and wrong.

---
## Beat 3 — Race it against a dumb competitor ✏️

Do we even need machine learning for this? Why not just
check whether the bill contains the word "china"?

Let's find out. Below is a **keyword model**: it says "about China" if any word from
your list appears in the text. We'll race it against the ML model on 12 tricky
sentences (some are the exact sneaky patterns you may have discovered in Beat 1).

**✏️ YOUR TURN:** run the cell as-is first to see the baseline race. Then edit the
list to make the keyword model smarter: add terms inside quotes, separated by commas.

In [ ]:
sneaky_terms = ["china", "chinese", "beijing"]   # <-- EDIT this list (add terms like "taiwan", "indo-pacific")

race_the_models(sneaky_terms)

> **What you should see** (with the starting list): a final score where the ML model
> wins comfortably — something like `Keyword model: 6/12  ML model: 11/12` — and a
> table showing exactly which sentences each one got right ✅ and wrong ❌.
> **If you get a red error:** check the list's shape — every term needs its own quotes,
> with commas between: `["china", "taiwan"]`.

### Things to try

1. Look at the table. *Why* did the keyword model miss what it missed? Find the
   sentence about "porcelain china tableware": what happened there, and why is that
   mistake the mirror image of the others?
2. Now upgrade the keyword list. Add `"taiwan"`, `"indo-pacific"`, `"byteDance"`...
   How high can you push its score?
3. Suppose you got it to 12/12. Here's the uncomfortable question: **next year's bills
   will use phrases nobody has written yet.** Your keyword list only knows the past.
   The ML model scored well on these sentences *without ever being shown them*,
   because it learned meaning, not vocabulary. That's the whole reason the field exists.

If you saved your "fooled it" sentences from Beat 1, which of the two models would
they fool? Reason it out from the table, then test your prediction with
`show_prediction(...)` in a scratch cell (Insert → Code cell).

---
## Bonus Beat — When the humans are wrong ✏️

Everything the model knows, it learned from **human-labeled examples**. So here's a
question almost nobody asks: what happens to our measurement of the model when the
*humans* make mistakes?

First, look at the held-out test bills and how humans labeled them. **▶️ JUST RUN:**

In [ ]:
# ▶️ JUST RUN — the test bills, numbered
show_test_bills()

> **What you should see:** a numbered table (rows 0, 1, 2, ...) of bill titles with
> their human labels.

Now the confusion matrix: a 2×2 scoreboard of where the model and the human labels
agree and disagree on those bills.

**✏️ YOUR TURN:** run it first with the empty list `[]` to see the honest scoreboard.
Then put a few row numbers from the table above into the list — that simulates a
tired annotator mislabeling those bills — and watch the scoreboard change.

In [ ]:
flip_these = []   # <-- EDIT: add row numbers from the table above, e.g. [2, 7, 15]

show_confusion_matrix(flip_these)

> **What you should see** (with `[]`): a blue 2×2 grid where the big numbers sit on
> the diagonal (both agree), small numbers off it, and a line like
> `Model and human labels agree on 24 of 27 bills.`
> **If you get a red error:** the list needs plain numbers with commas: `[2, 7]` —
> no quotes.

### Things to try

1. Flip 3 labels. The model didn't change *at all* — but its measured accuracy did.
   What does that tell you about trusting a single accuracy number?
2. Flip the labels of bills the model got *right*. Notice the accuracy drops even
   though the model's behavior is identical. We're now punishing correct answers.
3. Here's why this matters for **you, next week**: in Week 2 you'll be the human
   labeling real bills for the lab's classifier. Every label you write becomes part
   of what "correct" means. Careful annotators don't just help train the model —
   they make it *possible to know* whether the model works at all.

---
## Stretch goal (optional) — swap the brain

If you're comfortable and curious: the language model we've used
(`all-MiniLM-L6-v2`) is the small, fast one. There's a bigger, slower, smarter one.

**✏️ YOUR TURN:** change the name below and run. The download takes a few minutes.

In [ ]:
new_brain = "all-MiniLM-L6-v2"   # <-- EDIT: try "all-mpnet-base-v2"

rebuild_with(new_brain)

> **What you should see:** a download progress bar, then a new accuracy number.
> Is it higher? By a lot or a little? Was it worth a 5× bigger model?

After swapping, scroll back up and re-run Beats 1–3 with the new brain. Does it fall
for your sneaky sentences from Beat 1? (To go back to the fast model, change the name
back and run this cell again.)

---
## What you just did

With about ten lines of edited Python, you:

- trained a text classifier on real-style congressional bills,
- probed what it understands (and what fools it),
- discovered that the yes/no line is a *human choice*, not a model output,
- proved why learned meaning beats keyword matching,
- and saw that the model's report card is only as trustworthy as its human labels.

That last one is your bridge to **Week 2**: you'll be labeling real congressional
bills — the exact kind of data that trained the model you just spent four hours
interrogating. Now you know precisely why that work matters.

**Before you close this notebook:** write down (1) the single most surprising thing
the model did, and (2) your best sneaky sentence. Bring both to the Week 2 kickoff.